## Grounded answers from retrieved evidence

How strongly does evidence selection constrain a small local generator? This study compares lexical, semantic, and combined retrieval while holding answer generation fixed. It also tests evidence expansion, retry behavior, source attribution, and abstention. The released results describe bounded candidate evidence, with 64 held-out questions per dataset.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from advanced_rag.data import load
from advanced_rag.evaluation import aggregate, paired
from advanced_rag.pipeline import answer_scores, chunk
def read(name): return json.loads((ROOT / name).read_text(encoding='utf-8'))
metrics = read('reports/metrics.json')
config = read('configs/study.json')
frames = {name: pd.DataFrame(values).T for name, values in metrics.items()}
display(pd.DataFrame(config.items(), columns=['setting', 'value']))

## Research questions and controls

The comparisons ask whether retrieval gains carry through to answer F1, whether reranking earns its runtime, and whether expansion recovers evidence. Oracle evidence diagnoses selection errors. No-context generation checks how often answers come from the model despite absent evidence. Neither control is a deployable retrieval method.

In [ ]:
display(pd.DataFrame({'method': config['methods'], 'fixed_generator': ['FLAN-T5-small'] * len(config['methods'])}))

## Reference audit

Project 1 supplies retrieval methodology, validation principles, and notebook publication safeguards. Its local executed notebook and raw data were absent from this checkout. The file inventory records the unchanged reference bytes.

In [ ]:
audit = read('reports/reference-audit.json')
print('Reference cells:', audit['notebook_cells'], 'Files:', len(audit['files']))
print('Local executed reference available:', audit['local_executed_notebook_present'])

## Dataset decisions

HotpotQA provides annotated multi-passage supporting facts. SQuAD 2.0 supplies natural unanswerable questions. MuSiQue, 2WikiMultiHopQA, and Natural Questions were assessed and deferred in the research design. The selected releases have CC BY-SA 4.0 terms; attribution and original dataset links are in docs/research-design.md.

In [ ]:
display(pd.DataFrame(read('sources.lock.json'))[['name', 'license', 'bytes', 'sha256']])

## Acquisition provenance

The original HotpotQA CMU host failed DNS resolution during acquisition. The maintained Hugging Face dataset mirror was pinned by commit and its Parquet bytes checked against the published LFS hash. SQuAD came from the original Stanford project host. The converted HotpotQA JSON has a separate hash.

In [ ]:
display(pd.DataFrame(read('sources.lock.json'))[['name', 'url', 'acquired_utc']])

## Validate original annotations

Loading checks unique question IDs, answer spans, supporting-fact titles, and sentence indices. Out-of-range original sentence annotations are recorded without repair. Evaluation uses source-document labels, so these sentence defects do not change gold document sets. Counts refer to complete acquired development releases.

In [ ]:
datasets = {name: load(name, config['development_questions'], config['heldout_questions']) for name in metrics}
display(pd.DataFrame({name: info for name, (_, _, info) in datasets.items()}).T.drop(columns=['development_ids', 'heldout_ids', 'invalid_sentence_facts']))
print('Out-of-range sentence facts:', {name: len(info['invalid_sentence_facts']) for name, (_, _, info) in datasets.items()})

## Partition boundaries

SQuAD article groups are disjoint across local development and held-out samples. HotpotQA separates question IDs. Exact normalized question duplicates are removed from development. Pretrained model exposure to either benchmark remains unknown.

In [ ]:
for name, (dev, test, info) in datasets.items():
    assert not {r['group'] for r in dev} & {r['group'] for r in test}
    assert not {r['question'].strip().casefold() for r in dev} & {r['question'].strip().casefold() for r in test}
    print(name, len(dev), len(test), 'disjoint groups and questions')

## Answerability distribution

Unanswerable examples are original SQuAD annotations. No synthetic negative is included in the benchmark totals. The sample is selected by hash order, without balancing outcomes.

In [ ]:
display(pd.DataFrame([{'dataset': name, 'split': split, 'answerable': sum(bool(r['answers']) for r in rows), 'unanswerable': sum(not r['answers'] for r in rows)} for name, (dev, test, _) in datasets.items() for split, rows in [('development', dev), ('heldout', test)]]))

## Question types

HotpotQA distinguishes bridge and comparison questions. Their representation in a small subset affects which retrieval behaviors the experiment can detect.

In [ ]:
display(pd.Series([r['kind'] for r in datasets['hotpot'][1]]).value_counts().rename('heldout questions'))

## Candidate evidence

HotpotQA rankings are restricted to the supplied distractor set. SQuAD rankings use windows of the supplied paragraph. These two candidate regimes should not be pooled into a single retrieval score.

In [ ]:
display(pd.DataFrame([{'dataset': name, 'candidate_documents_mean': np.mean([len(r['docs']) for r in test]), 'annotated_sources_mean': np.mean([len(r['gold']) for r in test])} for name, (_, test, _) in datasets.items()]))

## Passage length exploration

Word counts describe source length before chunking. The generator uses its own tokenizer for input accounting; words are only a preprocessing unit.

In [ ]:
lengths = {name: [len(text.split()) for r in test for text in r['docs'].values()] for name, (_, test, _) in datasets.items()}
display(pd.DataFrame({name: pd.Series(v).describe() for name, v in lengths.items()}))
plt.figure(); plt.boxplot(list(lengths.values()), tick_labels=list(lengths)); plt.ylabel('Words per source'); plt.show()

## Chunk mapping

Windows contain at most 100 words with 20 words of overlap. Source IDs use a separate mapping, so repeated windows cannot inflate document recall. A long source can contribute several windows to the prompt.

In [ ]:
chunk_stats = []
for name, (_, test, _) in datasets.items():
    for r in test:
        docs, mapping = chunk(r['docs'])
        assert set(mapping.values()) == set(r['docs'])
        assert max(len(t.split()) for t in docs.values()) <= 100
        chunk_stats.append({'dataset': name, 'chunks': len(docs), 'documents': len(r['docs'])})
display(pd.DataFrame(chunk_stats).groupby('dataset').mean())

## Local model pins

MiniLM encodes windows, TinyBERT scores query-window pairs, and FLAN-T5-small generates answers. Immutable snapshot revisions and hashes identify all downloaded model files. The generator's small capacity limits the conclusions about larger systems.

In [ ]:
models = read('models.lock.json')
display(pd.DataFrame([{'role': r['role'], 'model': r['id'], 'revision': r['revision'], 'files': len(r['files'])} for r in models]))

## Executed environment

Generation is greedy with seed 42. This environment report records the actual device and source hashes used for the measurement pass.

In [ ]:
environment = read('reports/environment.json')
display({k: v for k, v in environment.items() if k not in ('source_sha256', 'config')})

## Answer scoring

Answer exact match removes punctuation, articles, and repeated whitespace. Token F1 counts token multiplicities and takes the best reference answer. Empty output scores correctly on an unanswerable item.

In [ ]:
assert answer_scores('The United States', ['United States']) == (1., 1.)
assert answer_scores('', []) == (1., 1.)
print('Answer normalization checks passed')

## Main HotpotQA results

All rows use identical held-out questions and the same generator. Compare answer F1 with context recall before attributing a gain to reasoning.

In [ ]:
display(frames['hotpot'][['n', 'em', 'f1', 'recall3', 'context_recall', 'p50_ms']])

## Main SQuAD results

Unanswerable questions contribute to exact match and F1 through empty outputs. A high abstention rate can hide poor answerable performance, so later analyses separate these groups.

In [ ]:
display(frames['squad'][['n', 'em', 'f1', 'abstained', 'abstention_correct', 'p50_ms']])

## Answer quality plot

The plot summarizes point estimates. The paired intervals below assess uncertainty in differences relative to hybrid retrieval.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (name, frame) in zip(axes, frames.items()):
    frame[['em', 'f1']].plot.bar(ax=ax); ax.set_title(name); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## Retained per-question results

Per-question answers, references, and candidate evidence stay in ignored local storage. Aggregate metrics are reconstructed from those original outputs, rather than typed into a report.

In [ ]:
scores = {name: read('data/experiments/' + name + '-scores.json') for name in metrics}
traces = {name: read('data/experiments/' + name + '-traces.json') for name in metrics}
for name in metrics:
    for method in metrics[name]: assert aggregate(scores[name][method]) == metrics[name][method]
print('Every aggregate equals the retained per-question scores')

## Retrieval and answer quality

Source recall and answer F1 can disagree because the prompt may omit a retrieved source, the generator may fail to use it, or an answer may be produced without supporting evidence. Correlation is descriptive and does not establish causation.

In [ ]:
for name in metrics:
    d = pd.DataFrame(scores[name]['hybrid'])
    print(name, d[['f1', 'context_recall']].corr().to_dict())
    display(d.groupby('context_recall', dropna=False)['f1'].agg(['count', 'mean']))

## Hybrid versus simpler retrieval

A positive paired F1 difference favors the method named in the row over hybrid. The fixed fusion setting was not optimized on held-out answers.

In [ ]:
statistics = read('reports/paired-bootstrap.json')
display(pd.DataFrame({name: {m: statistics[name][m] for m in ('bm25', 'dense')} for name in metrics}).T)

## Reranking cost

TinyBERT reranks at most ten candidates. These measurements include both query retrieval and generation, allowing comparison with the whole answer pipeline.

In [ ]:
display(pd.concat({name: frame.loc[['hybrid', 'rerank'], ['f1', 'context_recall', 'retrieval_ms', 'p50_ms', 'p95_ms']] for name, frame in frames.items()}))

## Evidence expansion

The multi-hop baseline expands the question using frequent terms in the top evidence window. It does not access gold answers or implement learned question decomposition. Expansion can drift toward a distractor.

In [ ]:
display(pd.concat({name: frame.loc[['hybrid', 'multihop'], ['f1', 'recall3', 'context_recall', 'p50_ms']] for name, frame in frames.items()}))

## Corrective retrieval

The retry rule checks an empty answer or answer-token overlap below 0.5. This rule can miss confidently unsupported answers and trigger on valid paraphrases. Retry totals include both generations.

In [ ]:
for name in metrics:
    entries = traces[name]['heldout'].values()
    print(name, 'retried:', sum(r['predictions']['corrective']['retried'] for r in entries), 'of', len(traces[name]['heldout']))
    display(frames[name].loc[['hybrid', 'corrective'], ['f1', 'p50_ms', 'input_tokens', 'output_tokens']])

## Context budget ablation

The smaller budget can skip windows that do not fit. The selector preserves whole windows and accounts for source labels. It may leave unused tokens, and a gold source can be retrieved but excluded.

In [ ]:
display(pd.concat({name: frame.loc[['hybrid', 'budget180'], ['f1', 'context_recall', 'input_tokens', 'p50_ms']] for name, frame in frames.items()}))

## Oracle evidence diagnostic

The oracle reorders windows by annotated source document. It can still select a window that misses the annotated sentence. Its advantage estimates part of the evidence-selection loss under this budget.

In [ ]:
display(pd.concat({name: frame.loc[['no_context', 'hybrid', 'oracle'], ['em', 'f1', 'context_recall']] for name, frame in frames.items()}))

## Development threshold selection

The evidence-overlap threshold was selected by development F1 before held-out scoring. The support score is not calibrated entailment probability.

In [ ]:
for name in metrics:
    selection = read('reports/' + name + '-selection.json')
    display(pd.DataFrame(selection['candidates'])[['threshold', 'em', 'f1', 'abstained']])
    print(name, 'selected:', selection['selected_threshold'])

## Selective prediction

Coverage is the fraction of nonempty answers. Selective EM evaluates only those answers. Unanswerable recall measures how often the system abstains on original unanswerable questions.

In [ ]:
display(pd.concat({name: frame.loc[['hybrid', 'abstention'], ['abstained', 'selective_em', 'unanswerable_recall', 'f1']] for name, frame in frames.items()}))

## Answerable and unanswerable breakdown

Separate strata help distinguish a useful abstention rule from a model that returns empty answers indiscriminately.

In [ ]:
for name in metrics:
    display(pd.DataFrame(scores[name]['abstention']).groupby('answerable')[['em', 'f1', 'abstained']].agg(['count', 'mean']))

## Citation label agreement

Citations are attached after generation using answer-token overlap. Precision and recall below compare source IDs with annotations. They do not verify that the answer follows from the cited passage.

In [ ]:
display(pd.concat({name: frame[['citation_precision', 'citation_recall']] for name, frame in frames.items()}))

## Answer-to-evidence diagnostic

Lexical overlap is retained to expose the abstention rule's behavior. Negation and comparison errors can have overlap 1.0. Manual support judgments are reported separately and must not be inferred from this plot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, name in zip(axes, metrics):
    d = pd.DataFrame(scores[name]['hybrid']); ax.scatter(d.support_overlap, d.f1, alpha=.5); ax.set_title(name); ax.set_xlabel('Lexical overlap'); ax.set_ylabel('Answer F1')
plt.tight_layout(); plt.show()

## Manual support review

A deterministic small review checks actual answer-to-cited-evidence consistency. This exploratory review has one reviewer and does not estimate population faithfulness. Review outcomes and explicit limitations remain separate from citation source-label metrics.

In [ ]:
review_path = ROOT / 'reports/manual-support-review.json'
review = json.loads(review_path.read_text()) if review_path.exists() else {'status': 'Review not completed; no faithfulness accuracy claim'}
display(review)

## Paired uncertainty

Percentile bootstrap intervals resample paired question F1 differences 2,000 times. They are exploratory, uncorrected for multiple comparisons, and do not account for shared SQuAD articles.

In [ ]:
for name in metrics:
    display(pd.DataFrame(statistics[name]).T)
    for method in statistics[name]:
        actual = paired([r['f1'] for r in scores[name]['hybrid']], [r['f1'] for r in scores[name][method]], config['bootstrap_replicates'])
        assert actual == statistics[name][method]
print('Bootstrap reports match retained outputs')

## Latency distributions

One sequential pass measures warm-process performance. Model loading and per-question indexing are separate. Method order, CPU frequency, and background tasks were not randomized.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, name in zip(axes, metrics):
    ax.boxplot([[r['total_ms'] for r in scores[name][m]] for m in metrics[name]], tick_labels=list(metrics[name])); ax.tick_params(axis='x', rotation=90); ax.set_title(name); ax.set_ylabel('Milliseconds')
plt.tight_layout(); plt.show()

## Memory and index construction

RSS is sampled after each question and includes all loaded models. GPU allocation is recorded only when CUDA is available. These observations cannot assign a memory peak to an individual method.

In [ ]:
display(pd.DataFrame({name: read('reports/' + name + '-resources.json') for name in metrics}).T)

## Token use and cost

No paid API is called. Token counts estimate local inference work; they do not measure electricity or hardware depreciation. Corrective totals include both attempts when a retry occurs.

In [ ]:
display(pd.concat({name: frame[['input_tokens', 'output_tokens', 'p50_ms']] for name, frame in frames.items()}))
print(environment['api_charges_usd'], environment['cost_scope'])

## Failure examples

The first three zero-F1 hybrid answers in deterministic held-out order are retained below for local inspection. Raw questions and answers disappear from the public notebook when outputs are stripped.

In [ ]:
for name in metrics:
    failures = []
    for qid, entry in traces[name]['heldout'].items():
        p = entry['predictions']['hybrid']
        if answer_scores(p['answer'], entry['references'])[1] == 0:
            failures.append({'id': qid, 'question': entry['question'], 'reference': entry['references'], 'answer': p['answer'], 'citations': p['citations']})
    display(pd.DataFrame(failures[:3]))

## Question-type errors

Bridge and comparison questions can fail for different reasons. Answer F1 by type is descriptive because the type counts are small and no per-type setting was selected.

In [ ]:
for name in metrics:
    rows = []
    for entry in traces[name]['heldout'].values():
        rows.append({'kind': entry['kind'], 'f1': answer_scores(entry['predictions']['hybrid']['answer'], entry['references'])[1]})
    display(pd.DataFrame(rows).groupby('kind').f1.agg(['count', 'mean']))

## Unsuccessful comparisons

Negative deltas remain part of the report. A more expensive stage that fails to improve held-out F1 has no demonstrated quality advantage in this sample.

In [ ]:
for name in metrics:
    baseline = frames[name].loc['hybrid', 'f1']
    delta = (frames[name]['f1'] - baseline).rename('F1 difference from hybrid')
    display(delta[delta < 0].to_frame())

## Prompt injection boundary

Retrieved passages are untrusted data. The prompt instructs the generator to ignore instructions in evidence, and the generator has no tools or command execution path. Prompt wording alone does not prove injection resistance. Adversarial checks remain a separate engineering check and are not included in benchmark scores.

In [ ]:
from advanced_rag.pipeline import Pipeline
import inspect
source = inspect.getsource(Pipeline.run)
assert 'untrusted quoted data' in source
print('Evidence is passed as prompt data; generation returns text only.')

## Publication equivalence

The executed notebook stays local with all outputs. Publication creates a deep copy and clears only that copy's outputs and counts. The verifier checks identical source cells and hashes the unchanged local file.

In [ ]:
print('Local output preservation and source equivalence are verified by scripts/publish_notebook.py after this clean execution.')
print('No raw documents, model weights, indexes, or traces are publication artifacts.')

## Conclusions and scope

These measurements apply to a small local generator and bounded evidence pools. Evidence expansion is a simple retrieval baseline, citation attachment is post hoc, and abstention depends on a weak lexical diagnostic. No result establishes general multi-hop reasoning, production safety, or open-domain performance. Larger samples, article-clustered uncertainty, stronger generators, and independent support annotation would be needed to extend these claims.

In [ ]:
for name in metrics:
    f = frames[name]
    print(name, 'hybrid F1:', round(f.loc['hybrid','f1'],4), 'BM25 F1:', round(f.loc['bm25','f1'],4), 'rerank F1:', round(f.loc['rerank','f1'],4))
    print('Oracle minus hybrid F1:', round(f.loc['oracle','f1'] - f.loc['hybrid','f1'],4))